# Task 6: Data Science Business Insights Dashboard (Netflix Dataset)

**Goal:** an end-to-end data science project that combines analytics, machine learning and business insights, delivered as an interactive dashboard and a professional report.

**Workflow**
1. Perform complete data analysis
2. Develop predictive models
3. Create interactive visualizations
4. Generate business recommendations
5. Present findings in a professional report

**Inputs:** `netflix_cleaned.csv` (Task 1) and `dashboard_template.html` (the dashboard's layout and JavaScript).
**Outputs:** `dashboard.html`, a single self-contained web page. Open it in any browser; it needs an internet connection only to load the Plotly charting library.

This task brings together the earlier work: the cleaning from Task 1, the trends from Task 2, the recommender from Task 3, the forecast from Task 4 and the classifier from Task 5.

## Setup

In [1]:
import json
import re
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

df = pd.read_csv("netflix_cleaned.csv", parse_dates=["date_added"]).reset_index(drop=True)
df["genres"] = df["listed_in"].str.split(", ")
print("Titles:", len(df))

Titles: 8786


## Step 1: Perform complete data analysis

### Headline metrics

In [2]:
full_years = df[df["year_added"].between(2016, 2020)]
kpis = {
    "titles": len(df),
    "movies": int((df["type"] == "Movie").sum()),
    "tv_shows": int((df["type"] == "TV Show").sum()),
    "countries": int(df.loc[df["country"] != "Unknown", "country"].nunique()),
    "genres": int(df["genres"].explode().nunique()),
    "peak_year": int(df.groupby("year_added").size().idxmax()),
    "peak_year_titles": int(df.groupby("year_added").size().max()),
    "median_movie_minutes": float(df["movie_minutes"].median()),
    "one_season_share": float((df["tv_seasons"] == 1).sum() / df["tv_seasons"].notna().sum()),
    "adult_share": float((df["audience"] == "Adults").mean()),
    "international_share": float((df.loc[df["country"] != "Unknown", "country"] != "United States").mean()),
}
pd.Series(kpis)

titles                  8786.000000
movies                  6123.000000
tv_shows                2663.000000
countries                 85.000000
genres                    42.000000
peak_year               2019.000000
peak_year_titles        2013.000000
median_movie_minutes      98.000000
one_season_share           0.672174
adult_share                0.455839
international_share        0.618779
dtype: float64

### Content mix over time
The TV share of new additions fell while movie additions boomed (2016 to 2018), then rose steadily. 2021 is a partial year (to September).

In [3]:
by_year = pd.crosstab(df["year_added"], df["type"]).loc[2016:2021]
by_year["TV share %"] = (by_year["TV Show"] / by_year.sum(axis=1) * 100).round(1)
by_year

type,Movie,TV Show,TV share %
year_added,,,
2016,251,175,41.1
2017,836,349,29.5
2018,1236,411,25.0
2019,1422,591,29.4
2020,1284,595,31.7
2021,993,505,33.7


### Markets
For each of the top 10 production countries: the movie/TV split, and growth in additions from 2016 to 2018 vs 2019 to 2021 (three years each; 2021 is partial, so growth is understated if anything).

In [4]:
known = df[df["country"] != "Unknown"]
top_markets = known["country"].value_counts().head(10).index
markets = pd.crosstab(known["country"], known["type"]).loc[top_markets]
markets["total"] = markets.sum(axis=1)
markets["TV share %"] = (markets["TV Show"] / markets["total"] * 100).round(0)

# Growth of each market: titles added in 2019 to 2021 vs 2016 to 2018
recent = known[known["year_added"].between(2019, 2021)]["country"].value_counts()
earlier = known[known["year_added"].between(2016, 2018)]["country"].value_counts()
markets["added 2016-18"] = earlier.reindex(top_markets).fillna(0).astype(int)
markets["added 2019-21"] = recent.reindex(top_markets).fillna(0).astype(int)
markets["growth %"] = ((markets["added 2019-21"] / markets["added 2016-18"] - 1) * 100).round(0)
markets

type,Movie,TV Show,total,TV share %,added 2016-18,added 2019-21,growth %
country,,,,,,,
United States,2395,845,3240,26.0,1094,2047,87.0
India,975,81,1056,8.0,503,553,10.0
United Kingdom,387,251,638,39.0,285,343,20.0
Pakistan,71,349,420,83.0,148,270,82.0
Canada,187,84,271,31.0,120,144,20.0
Japan,87,172,259,66.0,85,173,104.0
South Korea,49,165,214,77.0,80,134,68.0
France,148,65,213,31.0,80,129,61.0
Spain,129,53,182,29.0,78,103,32.0


**Observations**
- Outside the United States (+87%), the fastest-growing markets are Japan, Pakistan and South Korea, all mainly TV markets.
- India is the second-largest market but is over 90% film and grew only about 10%.

### Audience mix by year added

In [5]:
aud_mix = pd.crosstab(df["type"], df["audience"], normalize="index").mul(100).round(1)
aud_trend = pd.crosstab(df["year_added"], df["audience"], normalize="index").mul(100).round(1).loc[2016:2021]
aud_trend

audience,Adults,Little Kids,Older Kids,Teens,Unrated
year_added,,,,,
2016,41.5,4.9,22.8,24.4,6.3
2017,43.3,5.2,19.7,29.7,2.1
2018,47.2,5.3,16.0,30.6,0.9
2019,46.9,5.1,16.6,31.1,0.3
2020,45.7,9.2,15.3,29.9,0.0
2021,45.3,7.0,16.2,31.5,0.0


Adult-rated content holds steady at about 45% of additions. Little Kids content rose in 2020 (from about 5% to 9%), the year many families were at home.

## Step 2: Develop predictive models

Three models, each rebuilt here on the full dataset so the dashboard can use them. Model selection and detailed evaluation were done in Tasks 3 to 5; the same settings are reused.

### 2a. Content recommender (from Task 3)
TF-IDF features for genres, director, country, type, audience and decade, combined as a weighted average of per-feature cosine similarities. The six closest titles are precomputed for every title so the dashboard can show them instantly. Similarities are calculated in batches of 500 titles to keep memory low.

In [6]:
def to_tokens(value, split=True):
    if pd.isna(value):
        return ""
    items = str(value).split(",") if split else [str(value)]
    out = []
    for item in items:
        item = item.strip().lower()
        if item and item != "unknown":
            out.append(re.sub(r"[^a-z0-9]+", "_", item).strip("_"))
    return " ".join(out)

tok = pd.DataFrame({
    "genre": df["listed_in"].apply(to_tokens),
    "director": df["director"].apply(to_tokens),
    "country": df["country"].apply(to_tokens, split=False),
    "type": df["type"].apply(to_tokens, split=False),
    "audience": df["audience"].apply(to_tokens, split=False),
    "decade": (df["release_year"] // 10 * 10).astype(str) + "s",
})
WEIGHTS = {"genre": 1.0, "director": 0.6, "country": 0.5, "type": 0.5, "audience": 0.4, "decade": 0.2}
total_w = sum(WEIGHTS.values())
blocks = []
for col, w in WEIGHTS.items():
    Xb = TfidfVectorizer(token_pattern=r"[^ ]+", lowercase=False).fit_transform(tok[col])
    blocks.append(normalize(Xb) * np.sqrt(w / total_w))
X_rec = hstack(blocks).tocsr()

N_REC = 6
years = df["release_year"].values
rec_idx = np.zeros((len(df), N_REC), dtype=int)
rec_score = np.zeros((len(df), N_REC))
for start in range(0, len(df), 500):
    S = (X_rec[start:start + 500] @ X_rec.T).toarray()
    for k, i in enumerate(range(start, min(start + 500, len(df)))):
        sims = S[k]
        order = np.lexsort((np.abs(years - years[i]), -np.round(sims, 4)))
        order = order[order != i][:N_REC]
        rec_idx[i], rec_score[i] = order, sims[order]
print("Recommendations computed:", rec_idx.shape)

Recommendations computed: (8786, 6)


### 2b. Audience classifier (from Task 5)
Gradient boosting with the tuned settings from Task 5, predicting Kids, Teens or Adults.

To score **every** title fairly, predictions are made **out-of-fold**: the data is split into 5 parts, and each part is predicted by a model trained on the other 4. No title is scored by a model that saw it during training.

In [7]:
data_mask = (df["audience"] != "Unrated").values
lab = df[data_mask].copy()
lab["target"] = lab["audience"].replace({"Little Kids": "Kids", "Older Kids": "Kids"})
CLASSES = ["Kids", "Teens", "Adults"]

mlb = MultiLabelBinarizer().fit(lab["genres"])
top_countries = lab.loc[lab["country"] != "Unknown", "country"].value_counts().head(15).index
def features(part):
    g = pd.DataFrame(mlb.transform(part["genres"]), columns=mlb.classes_, index=part.index)
    c = pd.get_dummies(part["country"].where(part["country"].isin(top_countries), "Other"), prefix="c").astype(int)
    n = pd.DataFrame({
        "is_tv": (part["type"] == "TV Show").astype(int),
        "movie_minutes": part["movie_minutes"].fillna(0),
        "tv_seasons": part["tv_seasons"].fillna(0),
        "release_year": part["release_year"], "year_added": part["year_added"],
        "years_to_netflix": part["years_to_netflix"], "genre_count": part["genre_count"],
        "director_missing": part["director_missing"].astype(int),
    }, index=part.index)
    return pd.concat([g, c, n], axis=1)

X_cls, y_cls = features(lab), lab["target"]
model = HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=200,
                                       class_weight="balanced", random_state=RANDOM_STATE)
# Out-of-fold predictions: every title is scored by a model that never saw it
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
proba = cross_val_predict(model, X_cls, y_cls, cv=cv, method="predict_proba")
model_classes = sorted(CLASSES)                       # sklearn orders classes alphabetically
pred = np.array(model_classes)[proba.argmax(axis=1)]
lab["predicted"] = pred
for i, c in enumerate(model_classes):
    lab[f"p_{c}"] = proba[:, i]

cls_metrics = {
    "accuracy": accuracy_score(y_cls, pred),
    "macro_f1": f1_score(y_cls, pred, average="macro"),
    "baseline_accuracy": float(y_cls.value_counts(normalize=True).max()),
}
cm = confusion_matrix(y_cls, pred, labels=CLASSES)
print({k: round(v, 3) for k, v in cls_metrics.items()})
pd.DataFrame(cm, index=[f"actual {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES])

{'accuracy': 0.667, 'macro_f1': 0.665, 'baseline_accuracy': 0.46}


,pred Kids,pred Teens,pred Adults
actual Kids,1434,401,219
actual Teens,266,1558,821
actual Adults,222,973,2810


The out-of-fold results (about 67% accuracy against 46% for always guessing "Adults") match the Task 5 test set results closely, so the model's performance is stable.

### 2c. Rating review flags
A practical use of the classifier: list titles where the model is at least 70% confident the title belongs to a **very different** audience from its rating. Only the two costly directions are flagged:
- rated for families (Kids group) but the model sees adult content
- rated Adults but the model sees kids' content

In [8]:
# Rating review flags: the model is confident the title belongs to a very different audience.
# Only the two costly directions are flagged: rated Kids but looks Adult, or rated Adult but looks Kids.
flags = pd.concat([
    lab[(lab["target"] == "Kids") & (lab["p_Adults"] >= 0.70)].assign(flag="Rated for kids, looks adult", conf=lambda d: d["p_Adults"]),
    lab[(lab["target"] == "Adults") & (lab["p_Kids"] >= 0.70)].assign(flag="Rated adult, looks like kids' content", conf=lambda d: d["p_Kids"]),
]).sort_values("conf", ascending=False)
print("Titles flagged for rating review:", len(flags))
flags[["title", "type", "rating", "listed_in", "flag", "conf"]].head(10)

Titles flagged for rating review: 48


,title,type,rating,listed_in,flag,conf
3645,Fred Armisen: Standup For Drummers,Movie,TV-PG,"Music & Musicals, Stand-Up Comedy","Rated for kids, looks adult",0.901808
3628,Trailer Park Boys: Xmas Special,Movie,TV-MA,Movies,"Rated adult, looks like kids' content",0.875082
3991,Howards End,Movie,PG,"Classic Movies, Dramas, Independent Movies","Rated for kids, looks adult",0.869028
8345,Aussie Gold Hunters,TV Show,TV-MA,"Docuseries, International TV Shows, Reality TV","Rated adult, looks like kids' content",0.864056
1043,Four Sisters Before the Wedding,Movie,TV-MA,"Children & Family Movies, Comedies, Dramas","Rated adult, looks like kids' content",0.857982
3557,Kevin James: Never Don't Give Up,Movie,TV-PG,Stand-Up Comedy,"Rated for kids, looks adult",0.855950
1172,You're Everything To Me,Movie,TV-PG,"Comedies, Dramas, Independent Movies","Rated for kids, looks adult",0.849230
154,To All the Boys: Always and Forever - The Afte...,Movie,TV-MA,Movies,"Rated adult, looks like kids' content",0.837110
3008,Kevin James: Sweat the Small Stuff,Movie,TV-PG,Stand-Up Comedy,"Rated for kids, looks adult",0.832939
3721,Brian Regan: Nunchucks and Flamethrowers,Movie,TV-PG,Stand-Up Comedy,"Rated for kids, looks adult",0.825520


**Reading the flags.** Most flagged titles are probably rated correctly. They fall into clear patterns that are useful in themselves:
- **Clean stand-up comedians** (Kevin James, Brian Regan, Jim Gaffigan) rated TV-PG. Most stand-up is adult, so the model expects TV-MA.
- **Older PG films with horror themes** (Jaws 2, Jaws 3, The Car), from before the PG-13 rating existed in 1984.
- **Titles tagged only with the generic genre "Movies"**, mostly short specials and after-shows. With no real genre and a short runtime, the model guesses Kids. These point to a **metadata gap**, not a rating error.

### 2d. Content forecast (from Task 4)
The 12-month average model, which was the most accurate in Task 4's rolling backtest, with an 80% interval built from its actual backtest errors.

In [9]:
monthly = df.set_index("date_added").resample("MS").size()["2016-01":"2021-08"].astype(float)
H = 6
errors = []
for k in [4, 3, 2, 1]:
    c = len(monthly) - H * k
    errors.extend(monthly.iloc[c:c + H].values - monthly.iloc[:c][-12:].mean())
errors = np.array(errors)
point = monthly[-12:].mean()
lo, hi = point + np.percentile(errors, 10), point + np.percentile(errors, 90)
f_idx = pd.date_range("2021-09-01", "2022-12-01", freq="MS")
forecast = {
    "hist_x": [d.strftime("%Y-%m") for d in monthly.index], "hist_y": monthly.astype(int).tolist(),
    "fc_x": [d.strftime("%Y-%m") for d in f_idx], "point": round(point, 1), "lo": round(lo, 1), "hi": round(hi, 1),
    "annual_2022": int(round(point * 12)),
}
print(f"Forecast: {point:.0f} titles/month (80% interval {lo:.0f} to {hi:.0f}), about {point * 12:,.0f} in 2022")

Forecast: 164 titles/month (80% interval 125 to 222), about 1,973 in 2022


## Step 3: Create interactive visualizations

All results are packed into one JSON object and inserted into the dashboard template, producing a single `dashboard.html` file.

**Dashboard features**
- **Filters** for type, year added and country. Every overview chart and KPI tile updates instantly.
- **KPI tiles**: titles, movies, TV shows, countries, median movie length and share of one-season shows.
- **Charts**: titles added per year, top countries, top genres and audience mix, all with hover details.
- **Predictions**: the forecast with its 80% interval, and the classifier's accuracy and confusion matrix.
- **Find similar titles**: search any of the 8,786 titles and see its six closest matches with shared genres.
- **Rating review list**: the flagged titles, filterable by flag type.
- **Business recommendations**, each with its supporting evidence calculated from the data.
- **Responsive layout** (desktop, tablet and phone) and a dark mode.

In [10]:
# Package everything the dashboard needs into one JSON object
genre_list = sorted(df["genres"].explode().unique())
genre_pos = {g: i for i, g in enumerate(genre_list)}
country_list = df["country"].value_counts().index.tolist()
country_pos = {c: i for i, c in enumerate(country_list)}
aud_list = ["Little Kids", "Older Kids", "Teens", "Adults", "Unrated"]
aud_pos = {a: i for i, a in enumerate(aud_list)}

titles = [
    [r.title, int(r.type == "TV Show"), int(r.year_added), int(r.release_year),
     country_pos[r.country], aud_pos[r.audience], [genre_pos[g] for g in r.genres], int(r.duration_value)]
    for r in df.itertuples()
]

flag_rows = [
    [r.title, r.type, r.rating, r.listed_in, r.flag, round(float(r.conf), 2)]
    for r in flags.itertuples()
]

markets_out = [
    [c, int(m["Movie"]), int(m["TV Show"]), int(m["added 2016-18"]), int(m["added 2019-21"])]
    for c, m in markets.iterrows()
]

dashboard_data = {
    "generated_from": "netflix_cleaned.csv",
    "data_end": df["date_added"].max().strftime("%d %b %Y"),
    "genres": genre_list, "countries": country_list, "audiences": aud_list,
    "titles": titles,
    "rec": rec_idx.tolist(), "rec_score": np.round(rec_score, 2).tolist(),
    "forecast": forecast,
    "classifier": {**{k: round(v, 3) for k, v in cls_metrics.items()},
                   "classes": CLASSES, "cm": cm.tolist(), "n": int(len(lab))},
    "flags": flag_rows,
    "markets": markets_out,
    "kpis": {k: (round(v, 3) if isinstance(v, float) else v) for k, v in kpis.items()},
    "generic_movie_tag": int((df["listed_in"] == "Movies").sum()),
}

template = open("dashboard_template.html", encoding="utf-8").read()
html = template.replace("/*__DATA__*/null", json.dumps(dashboard_data, separators=(",", ":"), ensure_ascii=False))
with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"dashboard.html written ({len(html) / 1e6:.1f} MB)")

dashboard.html written (1.0 MB)


Open `dashboard.html` in a browser to explore it. To publish it online for free, upload it to a GitHub repository and turn on **GitHub Pages** (Settings, Pages); the page then has its own web address.

## Step 4: Generate business recommendations

Each recommendation is tied to evidence from the analysis. All of it is **catalog data** (what Netflix offers), not viewing data (what people watch), so every recommendation should be tested against viewing and subscriber data before acting on it.

| # | Recommendation | Evidence |
|---|---|---|
| 1 | **Keep shifting the mix toward series** | TV share of additions rose from 25% (2018) to 34% (2021). TV shows are typically added in their release year; movies a median of 2 years later. |
| 2 | **Grow international TV in fast-growing markets** | Outside the US, Japan (+104%), Pakistan (+82%) and South Korea (+68%) grew fastest, and are 66% to 83% TV. India is the gap: second-largest market, 8% TV, +10% growth. |
| 3 | **Plan for a stable volume, not growth** | Additions peaked at 2,013 in 2019 and settled at about 164 a month. The best model forecasts about 1,970 titles in 2022. |
| 4 | **Invest in metadata quality** | 29% of titles have no director, 53 carry only the generic genre "Movies", and there are no plot descriptions. These gaps limited both the recommender and the classifier. |
| 5 | **Use the classifier as a rating review aid** | 67% accurate overall; flags 48 of 8,704 rated titles, a list small enough to review by hand. |
| 6 | **Track why most series stop at one season** | 67% of TV shows have one season. The data cannot separate planned limited series from cancellations. |

## Step 5: Present findings in a professional report

The findings are written up in the accompanying report, **Netflix Content Strategy: Insights and Recommendations**, which covers the executive summary, method, findings, model results, recommendations and limitations.

## Limitations
- **Catalog data only.** No viewing, ratings or subscriber data, so the analysis shows what Netflix offers, not what succeeds.
- **Snapshot to September 2021.** Titles removed from Netflix before then are not in the data, and 2021 is a partial year.
- **One country per title** and **about 29% missing directors** limit market and creator analysis.
- **No plot descriptions or content tags**, which caps the recommender and the classifier.